# code

> fill kosha from the vault, and federate prose with code

In [ ]:
#| default_exp code

In [ ]:
#| hide
from nbdev.showdoc import *

kosha embeds identifiers, the vault embeds prose. They do not share a vector space, so `federate()`
fuses their *rankings* with RRF rather than their distances — the one thing that survives a change
of encoder.

In [ ]:
#| export
import warnings
import numpy as np
from fastcore.all import AttrDict, L, Path, patch, first
from litesearch.graph import rrf_all
from vishalakshi.core import Vault, tidy_bc

In [ ]:
#| export
LEGS = ('prose', 'repo', 'env', 'graph')

class EncoderShim:
    """Adapts a vault encoder to the `.encode(list)->ndarray` shape kosha expects.

    kosha builds `doc_encoder`/`query_encoder` around whatever `efn()` returns and calls
    `.encode()` on it for anything that is not a `FastEncode`, so this is the whole contract."""
    def __init__(self, enc): self.enc = enc
    def encode(self, texts, **kw): return np.asarray(self.enc(list(texts), **kw))
    def __repr__(self): return f'EncoderShim({getattr(self.enc, "__name__", "encoder")})'

In [ ]:
#| export
@patch
def kosha(self:Vault,
          dir=None,             # repo root; defaults to the vault's cwd repo
          share_encoder=None,   # None: kosha's code encoder, falling back to the vault's; True: force share
          **kw                  # forwarded to Kosha()
):
    """The `Kosha` instance for `dir`, cached on the vault.

    kosha indexes code with its own encoder (`potion-code-16M-v2`) because code embeds badly under
    a prose model — identifiers, not sentences. That is the right default and it is why federation
    below fuses *ranked lists* rather than vectors: the two stores do not share a vector space, so
    there is nothing to compare distance-wise. `share_encoder=True` forces one encoder across both
    (worse for code, but it makes the stores directly comparable), and is also the automatic
    fallback when kosha's model cannot be fetched."""
    from kosha.core import Kosha
    # `index_code(repo)` then `code_search(q)` must hit the *same* store. Without this, a bare
    # call resolves to kosha's own default root (the cwd repo) and silently searches nothing.
    dir = dir or getattr(self, '_code_dir', None)
    key = str(Path(dir).resolve()) if dir else '.'
    cache = getattr(self, '_kosha', None) or {}
    if key in cache: return cache[key]
    shim = lambda: EncoderShim(self.enc.doc)
    if share_encoder:
        k = Kosha(dir, efn=shim, **kw)
    else:
        try: k = Kosha(dir, **kw)
        except Exception as e:
            warnings.warn(f"kosha's code encoder is unavailable ({type(e).__name__}: {str(e)[:100]}); "
                          f"sharing the vault's {self.enc.method} encoder instead — code ranking "
                          f"will be worse than with a code-trained model")
            k = Kosha(dir, efn=shim, **kw)
    cache[key] = k
    self._kosha = cache
    return k

@patch
def index_code(self:Vault,
               dir=None,           # repo to index
               graph:bool=True,    # also build the AST call graph (callers, callees, PageRank)
               env:bool=False,     # also index installed packages (slow the first time)
               force:bool=False,
               verbose:bool=False,
               **kw
) -> dict:
    """Point the vault at a repo and fill kosha's stores from it.

    This is the code path proper — AST chunks, symbol names and a call graph — as opposed to
    `Vault.code()`, which files source files into the vault as ordinary documents. Use this one
    when you want symbol search and call-graph navigation; both can coexist, and `federate()`
    searches whichever exist."""
    k = self.kosha(dir, **kw)
    if dir: self._code_dir = str(Path(dir).resolve())   # remembered for later bare calls
    k.update_repo(dir, force=force, verbose=verbose)
    out = dict(repo=str(Path(dir or k.root).resolve()), chunks=len(list(k.code_st(select='id'))))
    if env:
        k.update_pkgs(k.status().get('stale_pkgs', {}), verbose=verbose, force=force)
        out['env_chunks'] = len(list(k.env_st(select='id')))
    if graph:
        k.graph.sync(dir=str(Path(dir or k.root)), force=force)
        out['graph_nodes'] = len(list(k.gn(select='node')))
        out['graph_edges'] = len(list(k.ge(select='caller')))
    return out

In [ ]:
#| export
@patch
def code_search(self:Vault, q:str, limit:int=10, repo:bool=True, env:bool=True,
                graph:bool=True, dir=None, **kw) -> L:
    """Search code through kosha: FTS + ANN over repo and environment, fused and rank-boosted.

    Supports kosha's `key:value` filters, so `'retry package:httpx'` and `'lang:.py chunker'` work."""
    return self.kosha(dir).context(q, limit=limit, repo=repo, env=env, graph=graph, **kw)

@patch
def symbol(self:Vault, name:str, depth:int=1, dir=None) -> AttrDict:
    'A symbol in the call graph: its file, PageRank and degree, plus its callers and callees.'
    k = self.kosha(dir)
    info = k.ni(name) or {}      # `ni`/`gn`/`ge` are kosha's graph accessors; node_info is not a method
    return AttrDict(node=name, info=dict(info), callers=L(info.get('callers') or []),
                    callees=L(info.get('callees') or []), neighbors=L(k.neighbors(name, depth)))

@patch
def where_to_add(self:Vault, description:str, limit:int=5, dir=None) -> L:
    'Where in the indexed repo a described change belongs — kosha ranking over the call graph.'
    return self.kosha(dir).where_to_add(description, limit=limit)

In [ ]:
#| export
def _prose_rows(v, q, limit, kind=None):
    'Vault sections, normalised to the federated row shape.'
    out = L()
    for s in v.sections(q, limit=limit, kind=kind):
        nid = s.get('node_id')
        out.append(AttrDict(source='prose', ref=nid, title=s.get('title') or '',
                            where=tidy_bc(s.get('breadcrumb')), score=s.get('score', 0.0),
                            text=' '.join(s.get('snippets') or [])[:600],
                            open=f'read({nid!r})'))
    return out

def _code_rows(rows, leg):
    'kosha hits, normalised to the federated row shape.'
    out = L()
    for r in L(rows):
        m = r.get('metadata') if isinstance(r, dict) else getattr(r, 'metadata', None)
        m = dict(m or {}) if not isinstance(m, dict) else m
        mod, path, ln = m.get('mod_name', ''), m.get('path', ''), m.get('lineno')
        txt = (r.get('content') if isinstance(r, dict) else getattr(r, 'content', '')) or ''
        out.append(AttrDict(source=leg, ref=mod or path, title=mod or Path(path).name if path else '',
                            where=f"{path}:{ln}" if path and ln else (path or mod),
                            score=(r.get('_rrf_score') if isinstance(r, dict) else 0.0) or 0.0,
                            text=txt[:600],
                            open=f"symbol({mod!r})" if mod else f"open {path}"))
    return out

def fed_rows(hits) -> list:
    'Flatten a federated result to plain dicts — for a CLI, an MCP payload or a frontend.'
    return [dict(n=i, source=h.source, title=h.title, where=h.where, ref=h.ref,
                 text=h.text, open=h.open) for i, h in enumerate(hits, 1)]

@patch
def federate(self:Vault,
             q:str,                # the query
             limit:int=12,         # fused hits returned
             prose:bool=True,      # the vault's own documents, papers, notes
             repo:bool=True,       # kosha's repo index
             env:bool=False,       # kosha's installed-package index
             kind=None,            # restrict the prose leg to some KINDS
             weights:dict=None,    # per-leg RRF weights, e.g. {'prose':1.0,'repo':1.5}
             dir=None,             # repo for the code legs
             per_leg:int=None,     # hits pulled from each leg before fusion
) -> AttrDict:
    """One ranked answer across prose and code, fused by RRF.

    The legs do not share a vector space — the vault embeds prose, kosha embeds identifiers — so
    they cannot be merged by distance. Reciprocal Rank Fusion only needs each leg's *ordering*,
    which is exactly what survives a change of encoder, and it is the same mechanism litesearch
    already uses to combine FTS with vectors. Each hit keeps its `source`, so a frontend can show
    where an answer came from, and `legs` reports what each contributed."""
    n = per_leg or max(limit, 10)
    ws, lists, legs = weights or {}, [], {}
    def leg(name, rows):
        rows = L(rows)
        if not rows: legs[name] = 0; return
        for i, r in enumerate(rows): r['_fid'] = f'{name}:{r.ref}:{i}' if not r.get('ref') else f'{name}:{r.ref}'
        legs[name] = len(rows); lists.append((name, rows))
    if prose: leg('prose', _prose_rows(self, q, n, kind=kind))
    if repo or env:
        try:
            k = self.kosha(dir)
            if repo: leg('repo', _code_rows(k.repo_context(q, limit=n), 'repo'))
            if env:  leg('env',  _code_rows(k.env_context(q, limit=n), 'env'))
        except Exception as e:
            legs['code_error'] = f'{type(e).__name__}: {str(e)[:120]}'
    if not lists: return AttrDict(query=q, hits=L(), legs=legs, note='no leg returned anything')
    fused = rrf_all([rows for _, rows in lists], limit=limit, id_key='_fid',
                    weights=[ws.get(name, 1.0) for name, _ in lists])
    hits = L(AttrDict(h) for h in fused)
    return AttrDict(query=q, hits=hits, legs=legs,
                    note=f"RRF over {', '.join(n_ for n_, _ in lists)}; "
                         f"legs use different encoders, so ranks are fused, not distances")